# Fine-Tuning and HF

In [1]:
import groq
import os
groq.api_key = os.getenv("GROQ_API_KEY")

In [2]:
!pip install jsonlines==4.0.0

In [6]:
!pip install datasets==2.20.0

### Preparing the dataset for fine-tuning

In [3]:
from datasets import load_dataset
import pandas as pd

dataset_view = load_dataset("sciq", split="train")

filtered_dataset = dataset_view.filter(lambda x: x["support"] != "" and x["correct_answer"] != "")
print("Number of questions with support: ", len(filtered_dataset))

Number of questions with support:  10481


In [4]:
df_view = pd.DataFrame(filtered_dataset)
columns_to_drop = ['distractor3', 'distractor1', 'distractor2']
df_view = df_view.drop(columns=columns_to_drop)
df_view.head()

,question,correct_answer,support
0,What type of organism is commonly used in prep...,mesophilic organisms,"Mesophiles grow best in moderate temperature, ..."
1,What phenomenon makes global winds blow northe...,coriolis effect,Without Coriolis Effect the global winds would...
2,Changes from a less-ordered state to a more-or...,exothermic,Summary Changes of state are examples of phase...
3,What is the least dangerous radioactive decay?,alpha decay,All radioactive decay is dangerous to living t...
4,Kilauea in hawaii is the world’s most continuo...,smoke and ash,Example 3.5 Calculating Projectile Motion: Hot...


In [5]:
import json
import jsonlines
import pandas as pd
from datasets import load_dataset

dataset = load_dataset("sciq", split="train")
filtered_dataset = dataset.filter(lambda x: x["support"] != "" and x["correct_answer"] != "")

df = pd.DataFrame(filtered_dataset)
columns_to_drop = ['distractor3', 'distractor1', 'distractor2']
df = df.drop(columns=columns_to_drop)

items = []
for idx, row in df.iterrows():
    detailed_answer = row['correct_answer'] + " Explanation: " + row['support']
    items.append({
        "messages": [
            {"role": "system", "content": "Given a science question, provide the correct answer with a detailed explanation."},
            {"role": "user", "content": row['question']},
            {"role": "assistant", "content": detailed_answer}
        ]
    })

with jsonlines.open('../Ch9/QA_prompts_and_completions.json', 'w') as writer:
    writer.write_all(items)

In [6]:
dfile="../Ch9/QA_prompts_and_completions.json"

In [7]:
import pandas as pd

df = pd.read_json(dfile, lines=True)
df

,messages
0,"[{'role': 'system', 'content': 'Given a scienc..."
1,"[{'role': 'system', 'content': 'Given a scienc..."
2,"[{'role': 'system', 'content': 'Given a scienc..."
3,"[{'role': 'system', 'content': 'Given a scienc..."
4,"[{'role': 'system', 'content': 'Given a scienc..."
...,...
10476,"[{'role': 'system', 'content': 'Given a scienc..."
10477,"[{'role': 'system', 'content': 'Given a scienc..."
10478,"[{'role': 'system', 'content': 'Given a scienc..."
10479,"[{'role': 'system', 'content': 'Given a scienc..."


### Fien-tuning the model

In [22]:
!pip install --upgrade groq

  Using cached groq-1.1.2-py3-none-any.whl.metadata (16 kB)
Using cached groq-1.1.2-py3-none-any.whl (141 kB)


In [31]:
import openai
api_key = os.getenv("OPENAI_API_KEY")
openai.api_key = api_key

In [ ]:
from openai import OpenAI
import jsonlines
client = OpenAI(api_key=api_key)

result_file = client.files.create(
  file=open("QA_prompts_and_completions.json", "rb"),
  purpose="fine-tune"
)

print(result_file)
param_training_file_name = result_file.id
print(param_training_file_name)


ft_job = client.fine_tuning.jobs.create(
  training_file=param_training_file_name,
  model="gpt-4o-mini-2024-07-18"
)

print(ft_job)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
import pandas as pd
from openai import OpenAI
client = OpenAI()
response = client.fine_tuning.jobs.list(limit=3) 

job_ids = []
created_ats = []
statuses = []
models = []
training_files = []
error_messages = []
fine_tuned_models = []  # List to store the fine-tuned model names

for job in response.data:
    job_ids.append(job.id)
    created_ats.append(job.created_at)
    statuses.append(job.status)
    models.append(job.model)
    training_files.append(job.training_file)
    error_message = job.error.message if job.error else None
    error_messages.append(error_message)

    fine_tuned_model = job.fine_tuned_model if hasattr(job, 'fine_tuned_model') else None
    fine_tuned_models.append(fine_tuned_model)

df = pd.DataFrame({
    'Job ID': job_ids,
    'Created At': created_ats,
    'Status': statuses,
    'Model': models,
    'Training File': training_files,
    'Error Message': error_messages,
    'Fine-Tuned Model': fine_tuned_models  
})

df['Created At'] = pd.to_datetime(df['Created At'], unit='s')
df = df.sort_values(by='Created At', ascending=False)
df

In [ ]:
import pandas as pd

generation=False  # until the current model is fine-tuned
non_empty_models = df[df['Fine-Tuned Model'].notna() & (df['Fine-Tuned Model'] != '')]

if not non_empty_models.empty:
    first_non_empty_model = non_empty_models['Fine-Tuned Model'].iloc[0]
    print("The latest fine-tuned model is:", first_non_empty_model)
    generation=True
else:
    first_non_empty_model='None'
    print("No fine-tuned models found.")

### Using the fine-tuned model

In [ ]:
prompt = "What phenomenon makes global winds blow northeast to southwest or the reverse in the northern hemisphere and northwest to southeast or the reverse in the southern hemisphere?"

In [ ]:
if generation==True:
    response = client.chat.completions.create(
        model=first_non_empty_model,
        temperature=0.0,  
        messages=[
            {"role": "system", "content": "Given a question, reply with a complete explanation for students."},
            {"role": "user", "content": prompt}
        ]
    )
else:
    print("Error: Model is None, cannot proceed with the API request.")

In [ ]:
if generation==True:
  print(response)

In [ ]:
if (generation==True):
  response_text = response.choices[0].message.content
  print(response_text)

In [ ]:
import textwrap

if generation==True:
  wrapped_text = textwrap.fill(response_text.strip(), 60)
  print(wrapped_text)